# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

> [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

This notebook demonstrates step-by-step data loading, overview, processing, and visualization, referencing all entities by their `@id`.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', 'unknown')}")
print(f"Identifier: {getattr(metadata, 'identifier', 'unknown')}")
print(f"Version: {getattr(metadata, 'version', 'unknown')}")
print(f"License: {getattr(metadata, 'license', 'unknown')}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

The dataset may contain multiple record sets. We'll list all record sets along with their `@id`, name, and description. We will then preview the first few records from each record set (using their `@id`).

Let's first obtain the list of record sets.

In [ ]:
# Get all record sets by @id
record_sets = dataset.record_sets()

print("Available Record Sets:")
for rs in record_sets:
    print(f"\n@id: {rs.id}\nName: {rs.name}\nDescription: {getattr(rs, 'description', 'No description')}")

# Preview first record from each record set
for rs in record_sets:
    print(f"\nRecords from Record Set @id: {rs.id}")
    for i, record in enumerate(dataset.records(record_set=rs.id)):
        print(record)
        if i >= 2:  # preview up to 3 records
            break


Let's also list all fields (columns) for each record set, referencing them by their `@id`.

In [ ]:
for rs in record_sets:
    print(f"\nRecordSet Name: {rs.name} (@id: {rs.id})")
    print("Fields (columns):")
    for field in rs.fields:
        print(f"- @id: {field.id}, name: {field.name}, type: {field.data_type}")


## 3. Data Extraction
Load data from each record set into a Pandas DataFrame for analysis. Use the record set and field `@id`s from the overview.

We'll demonstrate extraction for all available record sets (using their `@id`).

In [ ]:
# Extract data from each record set
dataframes = {}

for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"\nColumns in DataFrame for RecordSet @id: {rs.id}")
    print(df.columns.tolist())
    print(df.head())


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll select a numeric field and a categorical grouping field from one of the record sets. For illustration, we identify a numeric field (e.g., 'Age') and a group field (e.g., 'Sex') using their `@id`.

In [ ]:
# For demonstration, select the main record set (first one):
if len(record_sets) > 0:
    primary_rs = record_sets[0]
    df = dataframes[primary_rs.id]
    # Find a numeric field and a categorical field by data type
    numeric_field_ids = [field.id for field in primary_rs.fields if field.data_type in ['schema:Number', 'schema:Integer', 'schema:Float']]
    group_field_ids = [field.id for field in primary_rs.fields if field.data_type in ['schema:Text', 'schema:Boolean'] and ('sex' in field.name.lower() or 'gender' in field.name.lower())]
    # Use first found numeric field and group field
    if numeric_field_ids:
        numeric_field_id = numeric_field_ids[0]
    else:
        numeric_field_id = None
    if group_field_ids:
        group_field_id = group_field_ids[0]
    else:
        group_field_id = None

    if numeric_field_id and numeric_field_id in df.columns:
        threshold = 50  # Demo threshold for age (change if needed)
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        if group_field_id and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No suitable numeric field found for EDA.")
else:
    print("No record sets available in dataset.")


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

For demonstration, we'll plot the distribution of the numeric field and visualize grouping by the group field (if available).

In [ ]:
import matplotlib.pyplot as plt

if len(record_sets) > 0 and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(7, 4))
    df[numeric_field_id].hist(bins=15)
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(7, 4))
        df.boxplot(column=numeric_field_id, by=group_field_id)
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.suptitle('')
        plt.show()


## 6. Conclusion
This notebook demonstrated how to load, overview, extract, and visualize tabular data from the FAIR^2 dataset using the `mlcroissant` library. All dataset entities (record sets, fields, etc.) were referenced by their `@id` as per FAIR and Croissant standards.

Key exploration steps included:
- Listing available record sets and their fields via `@id`
- Loading each record set into a Pandas DataFrame
- Filtering and normalizing a numeric field, grouping by a categorical field
- Visualizing distributions and relationships

For further analysis, refer to the schema documentation and explore additional record sets and fields present in the Croissant metadata.